# 01 — SPX option data and SVI surface fit

Pull one trading day of SPX option chain via OpenBB, filter for liquid call quotes, invert Black-Scholes to recover implied volatilities, and fit a raw-SVI slice to each maturity. The output is an arbitrage-checked surface used downstream by calibration.

In [ ]:
import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from volengine.backtesting import load_option_chain, filter_for_calibration
from volengine.surfaces import fit_svi_slice, svi_implied_vol, SVISurface, implied_vol

## Load and filter

The loader caches each (symbol, date) to a parquet under `data/cache/` so the notebook is reproducible without re-hitting OpenBB.

In [ ]:
snapshot = load_option_chain('SPY', dt.date.today(), provider='yfinance')
filtered = filter_for_calibration(snapshot, moneyness_band=(0.85, 1.15))
print(f'Loaded {len(snapshot.chain)} raw quotes, kept {len(filtered)} after liquidity filter.')
filtered.head()

## Invert mid prices to implied vols

In [ ]:
ivs = []
for _, row in filtered.iterrows():
    iv = implied_vol(price=row['mid'], S0=snapshot.spot, K=row['strike'],
                     T=row['dte_years'], r=snapshot.r, q=snapshot.q, flag='call')
    ivs.append(iv)
filtered = filtered.assign(iv_inverted=ivs).dropna(subset=['iv_inverted'])
filtered.groupby('expiration')['iv_inverted'].agg(['count', 'mean', 'std'])

## Fit raw-SVI to each maturity slice

In [ ]:
slices = {}
for T, group in filtered.groupby('dte_years'):
    if len(group) < 5:
        continue
    F = snapshot.spot * np.exp((snapshot.r - snapshot.q) * T)
    k = np.log(group['strike'].values / F)
    iv = group['iv_inverted'].values
    slices[T] = fit_svi_slice(k, iv, T)
surface = SVISurface(maturities=np.array(sorted(slices.keys())), slices=slices)
print(f'Fitted {len(slices)} slices. Calendar-arb-free: {surface.check_calendar_arbitrage()}')

## Overlay plot: market IVs vs. SVI fit

In [ ]:
n_T = len(slices)
fig, axes = plt.subplots(1, n_T, figsize=(4 * n_T, 4), sharey=True)
axes = [axes] if n_T == 1 else axes
for ax, (T, p) in zip(axes, sorted(slices.items())):
    group = filtered[filtered['dte_years'] == T]
    F = snapshot.spot * np.exp((snapshot.r - snapshot.q) * T)
    k_data = np.log(group['strike'].values / F)
    ax.scatter(k_data, group['iv_inverted'], s=15, label='market')
    k_grid = np.linspace(k_data.min(), k_data.max(), 100)
    ax.plot(k_grid, svi_implied_vol(k_grid, T, p), 'r-', label='SVI fit')
    ax.set_title(f'T = {T:.3f}y')
    ax.set_xlabel('log-moneyness k')
    ax.legend()
axes[0].set_ylabel('implied vol')
fig.tight_layout()
fig.savefig('../results/figures/svi_fit.png', dpi=120)
plt.show()